# 🎬 Render "PremiumReel" (Your Time Is Running Out)

This notebook renders the Remotion composition **PremiumReel** (1080×1920, 30 fps, 15 s, H.264 MP4) entirely in Google Colab — nothing runs on your own PC.

**How to use:** `Runtime → Run all`. When it finishes, the MP4 downloads automatically (allow pop-ups/downloads if your browser asks).

Typical time on free Colab: about 5–10 minutes total (most of it is installing).

## 1 · Settings
Change `BRANCH` only if the pull request is **not merged yet** (use the PR branch name). After merging, keep `main`.

In [ ]:
REPO_URL = "https://github.com/santosarder477-maker/New-one.git"
BRANCH = "main"          # e.g. "ccr-b4755598-tfagsh" before the PR is merged
COMPOSITION = "PremiumReel"
OUTPUT = "out/PremiumReel.mp4"
CRF = 18                 # lower = higher quality / bigger file (18 is visually lossless-ish)

## 2 · Install Node.js 22 and Chrome system libraries

In [ ]:
%%bash
set -e
if ! node -v 2>/dev/null | grep -q '^v22'; then
  curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null
  apt-get install -y nodejs > /dev/null
fi
echo "Node: $(node -v)  npm: $(npm -v)"

# Libraries required by Remotion's headless Chrome on Ubuntu.
apt-get update -qq > /dev/null
for pkg in libnss3 libdbus-1-3 libatk1.0-0 libgbm-dev libasound2 libasound2t64 libxrandr2 \
           libxkbcommon-dev libxfixes3 libxcomposite1 libxdamage1 libatk-bridge2.0-0 \
           libpango-1.0-0 libcairo2 libcups2; do
  apt-get install -y -qq "$pkg" > /dev/null 2>&1 || echo "(skipped $pkg — not available on this Ubuntu version)"
done
echo "System libraries ready."

## 3 · Download the project from GitHub
If your repository is **private**, add a Colab secret named `GITHUB_TOKEN` (🔑 icon in the left sidebar) containing a GitHub token with read access.

In [ ]:
import os, shutil, subprocess

url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass  # no secret set -> public clone

shutil.rmtree("/content/reel", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, "/content/reel"], check=True)
os.chdir("/content/reel")
print("Project ready at", os.getcwd())

## 4 · Install project dependencies and the render browser

In [ ]:
%%bash
cd /content/reel
npm ci --no-audit --no-fund
npx remotion browser ensure

## 5 · Render the video (H.264 MP4)

In [ ]:
import subprocess
cmd = [
    "npx", "remotion", "render", COMPOSITION, OUTPUT,
    "--codec=h264", f"--crf={CRF}", "--pixel-format=yuv420p",
    "--timeout=120000",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True, cwd="/content/reel")

## 6 · Preview and download

In [ ]:
import os
from base64 import b64encode
from IPython.display import HTML, display

path = os.path.join("/content/reel", OUTPUT)
print(f"File: {path}  ({os.path.getsize(path)/1e6:.1f} MB)")

data = b64encode(open(path, "rb").read()).decode()
display(HTML(f'<video width="270" controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))

from google.colab import files
files.download(path)

### Troubleshooting
- **Download didn't start:** open the 📁 Files panel (left sidebar) → `reel/out/PremiumReel.mp4` → ⋮ → *Download*.
- **`Remote branch main not found`:** the PR isn't merged yet — set `BRANCH` to the PR branch in step 1 and re-run.
- **Timeout during render:** re-run step 5; free Colab machines are sometimes slow to start Chrome.
- **Want to save to Google Drive instead:** `from google.colab import drive; drive.mount('/content/drive')` then copy the MP4 into `/content/drive/MyDrive/`.